In [ ]:
! pip install nltk gensim scikit-learn torch pandas

In [ ]:
# don't run it
# !curl -L -o ./fake-news-detection-datasets.zip https://www.kaggle.com/api/v1/datasets/download/emineyetm/fake-news-detection-datasets

In [ ]:
# read system-component-diag.txt from the project root and print its contents
# create a variable called path and assign it the value of the path to the system-component-diag.txt file
import pathlib
path = f'{pathlib.Path.cwd()}/system-component-diag.txt'
# use this path in the open function to read the contents of the file and print it
with open(path,'r') as file:
    contents = file.read()
    print(contents)


In [ ]:
# read system-component-diag.txt from the project root and print its contents
# create a variable called path and assign it the value of the path to the system-component-diag.txt file
import pathlib
path = f'{pathlib.Path.cwd()}/more-detailed-arch.txt'
# use this path in the open function to read the contents of the file and print it
with open(path,'r') as file:
    contents = file.read()
    print(contents)

In [ ]:
import pathlib
path = f'{pathlib.Path.cwd()}/final_design.txt'
with open(path, 'r') as file:
    contents = file.read()
    print(contents)

In [ ]:
! pip install -U spacy

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import re

# Download necessary NLTK assets once
nltk.download('punkt')
nltk.download('stopwords')

def master_preprocessor(raw_text):
    # Raw text for SpaCy (POS Tagging & Dependency Parsing)
    # No modifications needed, SpaCy handles its own tokenization internally.
    spacy_input = raw_text.strip()
    lowercased_text = raw_text.lower()
    
    # tokenization for neuralLM and HMM
    tokens_with_stopwords = word_tokenize(lowercased_text)
    cleaned_words = [ word for word in tokens_with_stopwords if word not in string.punctuation]
    
    # Flavor 3: Tokenization WITHOUT Stop Words (For Word2Vec & Cosine Similarity)
    stop_words = set(stopwords.words('english'))
    tokens_without_stopwords = [word for word in cleaned_words if word not in stop_words]
    
    return {
        "spacy_raw_string": spacy_input,     # spacy_input for pos_tagging
        "sequential_tokens": cleaned_words, # for neural lm and HMM
        "semantic_tokens": tokens_without_stopwords # for GloVe, Word2Vec
    }

#test
print(master_preprocessor("I am in london today, I do not know how much I can cover"))

In [ ]:
# Neural LM task 3, above is just some sample from here - https://drive.google.com/file/d/1RmoBFVsOsmXD03i2zJnRnHKbs2VtrIcR/view
# vipin is woking here
# I have fake.csv and true.csv
# take 20% from both as test and 80% to train the network
# Steps: 
# 1. build vocabulary over corpus - say 10000 +1 (<UNK>), 
# 2. build feed forward nn architecture with nn.Module, check if we can place already built weights incase we need to test it against benchmark embeddings
# 3. build training loop over context window and compute loss from the target word .
# for example 1. context window - 3, 2. sentence = we have visited europe, asia and other continents
# context window words - we have visited , target_word - europe and it will slide through out the sentence, 
# we need to do it for all the sentences
# you're asking, how do we map it to vocabulary ? - vocab number will be fed to NLM
# 4. take the perplexity scoring for each sentence by taking out average for each sentence
# 5. prepare original datasets (fake and real) with their perplexity scoring
# 6. feed to logistic regression

In [ ]:
# Step1: prepare test and train data
! wc -l fake-news-detection-datasets/News\ _dataset/Fake.csv
! wc -l fake-news-detection-datasets/News\ _dataset/True.csv

import pandas as pd
from pandas import DataFrame
fake_news_df: DataFrame = pd.read_csv("fake-news-detection-datasets/News _dataset/Fake.csv")
true_news_df: DataFrame = pd.read_csv("fake-news-detection-datasets/News _dataset/True.csv")
fake_news_df["full_content"] = fake_news_df['title'].fillna('')+ ". "+ fake_news_df["text"].fillna('')
fake_news_df['is_fake'] = 1
true_news_df['full_content'] = true_news_df['title'].fillna('')+". "+true_news_df['text'].fillna('')
true_news_df['is_fake'] = 0

training_fake_df: DataFrame = fake_news_df.sample(frac=.8, random_state=42)
training_true_df: DataFrame = true_news_df.sample(frac=.8, random_state=42)

testing_fake_df: DataFrame = fake_news_df.drop(training_fake_df.index)
testing_true_df: DataFrame = true_news_df.drop(training_true_df.index)

training_df = pd.concat([training_true_df, training_fake_df]).sample(frac=1, random_state=42).reset_index(drop=True)
testing_df = pd.concat([testing_true_df, testing_fake_df]).sample(frac=1, random_state=42).reset_index(drop=True)

training_df['full_content_arr'] = training_df['full_content'].apply(lambda x: master_preprocessor(x)['sequential_tokens'])
testing_df['full_content_arr'] = testing_df['full_content'].apply(lambda x: master_preprocessor(x)['sequential_tokens'])

print("Training set counts:\n", training_df['is_fake'].value_counts())
print("Testing set counts:\n", testing_df['is_fake'].value_counts())

In [ ]:
training_df[training_df['is_fake']==1].head()

In [ ]:
# Step2: build vocabulary over training text corpus

from collections import Counter
from pandas import DataFrame

# Step 2: build vocabulary over training text corpus
def build_vocab(training_df: DataFrame, max_size=5000):
    # 1. Count frequencies of all tokens across the entire training column
    token_counts = Counter()
    for tokens_list in training_df['full_content_arr']:
        if isinstance(tokens_list, list): # Safety check
            token_counts.update(tokens_list)
            
    # 2. Extract the most frequent words up to max_size - 1 
    # We leave 1 slot open for the special <UNK> token
    most_common_words = token_counts.most_common(max_size - 1)
    
    # 3. Initialize your dictionaries with the Unknown token anchor
    UNKNOWN = "<UNK>"
    word_to_idx: dict = {UNKNOWN: 0}
    idx_to_word: dict = {0: UNKNOWN}
    
    # 4. Correctly assign incremental unique indices to the top words
    for idx, (word, count) in enumerate(most_common_words, start=1):
        word_to_idx[word] = idx
        idx_to_word[idx] = word
        
    # 5. Compute actual final vocabulary size
    vocab_size = len(word_to_idx)
    
    return word_to_idx, idx_to_word, vocab_size

word_to_idx, idx_to_word, vocab_size = build_vocab(training_df.head())



In [ ]:
print(word_to_idx)

In [ ]:
# 3. make feed forward neural network

#3.1 generate context window

import torch
import torch.nn as nn
import torch.optim as optim



def generate_window_dataset(df_column, word_to_idx, context_size=3):
    X_list = []
    Y_list = []
    
    for tokens in df_column:
        if not isinstance(tokens, list) or len(tokens) <= context_size:
            continue
            
        # 1. Map string tokens to vocabulary integer IDs
        # If a word isn't in your 10,000 dictionary, default to index 0 (<UNK>)
        idx_sequence = [word_to_idx.get(token, 0) for token in tokens]
        
        # 2. Extract context-history slices and the target word
        for i in range(len(idx_sequence) - context_size):
            context = idx_sequence[i : i + context_size]
            target = idx_sequence[i + context_size]
            
            X_list.append(context)
            Y_list.append(target)
            
    # 3. Convert to PyTorch tensors
    X_tensor = torch.tensor(X_list, dtype=torch.long)
    Y_tensor = torch.tensor(Y_list, dtype=torch.long)
    
    return X_tensor, Y_tensor

# Generate inputs and labels using your verified vocab from Step 2
X_train, Y_train = generate_window_dataset(training_df['full_content_arr'], word_to_idx, context_size=3)

print("Shape of X inputs entering PyTorch:", X_train.shape) # Expected: (Total_Windows, 3)
print("Shape of Y labels entering PyTorch:", Y_train.shape)



In [ ]:
# 3.2
# make it compatible with pretrained context embeddings as well, like GloVe or Word2Vec

class FeedforwardNeuralLM(nn.Module):
    def __init__(self, vocab_size=5000, embedding_dim=100, context_size=3, hidden_dim=32):
        super(FeedforwardNeuralLM, self).__init__()
        
        # 1. Embedding Layer: Maps word integer IDs to dense vectors
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. Hidden Layer: Takes concatenated vectors and applies non-linearity
        # Input size is: context_size * embedding_dim (e.g., 3 words * 100 dim = 300)
        self.linear1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.activation = nn.Tanh() # or nn.ReLU() as per your slides
        
        # 3. Output Layer: Projects hidden layer back to vocabulary distribution
        self.linear2 = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, inputs):
        # inputs shape: (batch_size, context_size)
        
        # Look up embeddings: shape becomes (batch_size, context_size, embedding_dim)
        embeds = self.embeddings(inputs)
        
        # Concatenate the context embeddings into one long vector per sample
        # shape becomes: (batch_size, context_size * embedding_dim)
        flattened_embeds = embeds.view(inputs.size(0), -1)
        
        # Pass through hidden layer
        hidden = self.activation(self.linear1(flattened_embeds))
        
        # Pass through output layer to get raw scores (logits) for every word in vocabulary
        output = self.linear2(hidden)
        return output
    


In [ ]:
#3.3
from torch.utils.data import DataLoader, TensorDataset

model = FeedforwardNeuralLM(vocab_size=vocab_size,hidden_dim=64)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=.1)
num_epoch = 5

train_dataset = TensorDataset(X_train, Y_train)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
# for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.train()
for epoch in range(num_epoch):
    epoch_loss = 0.0
    for X, Y in train_loader:
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, Y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"total loss in epoch_{epoch}: {epoch_loss}, average loss: {epoch_loss/len(train_loader):.4f}")



In [ ]:
# Step4: evaluate using perplexity
import numpy as np

def calculate_article_perplexity(tokens_list, model, word_to_idx, context_size=3):
    # If the article is too short to even fill one context window, assign a fallback score
    if not isinstance(tokens_list, list) or len(tokens_list) <= context_size:
        return 1000.0  #
    
    # 1. Convert string tokens to vocabulary integer IDs
    idx_sequence = [word_to_idx.get(token, 0) for token in tokens_list]
    
    X_list = []
    Y_list = []
    
    # 2. Extract windows just for this single article
    for i in range(len(idx_sequence) - context_size):
        X_list.append(idx_sequence[i : i + context_size])
        Y_list.append(idx_sequence[i + context_size])
        
    # Convert to PyTorch tensors
    X_tensor = torch.tensor(X_list, dtype=torch.long).to(device)
    Y_tensor = torch.tensor(Y_list, dtype=torch.long).to(device)
    
    # Use reduction='none' so PyTorch returns the individual loss value for EACH window,
    # rather than automatically averaging everything across the batch.
    criterion = nn.CrossEntropyLoss(reduction='none')
    
    # Switch model to evaluation mode (turns off dropout, gradient tracking, etc.)
    model.eval()
    with torch.no_grad():
        # Get raw prediction logits from your trained model
        outputs = model(X_tensor) # Shape: (Num_Windows, Vocab_Size)
        
        # Calculate loss for every single window in this specific article
        individual_losses = criterion(outputs, Y_tensor) # Shape: (Num_Windows,)
        
        # 3. Calculate the mathematical average loss for this individual document
        avg_doc_loss = torch.mean(individual_losses).item()
        
        # 4. Compute Perplexity: e^(average loss)
        perplexity = np.exp(avg_doc_loss)
        
    return perplexity

In [ ]:
#Step 5:  Apply the perplexity function to your training dataset
print("Extracting perplexity scores for Training Data...")
training_df['perplexity'] = training_df['full_content_arr'].apply(
    lambda tokens: calculate_article_perplexity(tokens, model, word_to_idx, context_size=3)
)

# Apply the exact same perplexity function to your testing dataset
print("Extracting perplexity scores for Testing Data...")
testing_df['perplexity'] = testing_df['full_content_arr'].apply(
    lambda tokens: calculate_article_perplexity(tokens, model, word_to_idx, context_size=3)
)

# Take a quick peek at the mapped results
print(training_df[['full_content', 'is_fake', 'perplexity']].head())